In [125]:
from bs4 import BeautifulSoup
import requests

from warnings import filterwarnings
filterwarnings('ignore')

url = "https://tftactics.gg/champions/"
req = requests.get(url)
champ_reflist = []
soup = BeautifulSoup(req.text, "html.parser")
for link in soup.find_all('a'):
    a = link.get('href')
    if "/champions/" in a:
        champ_reflist.append(a[11:]) 
  
champ_reflist = champ_reflist[2:]


stat_list = {"Cost","Health", "Mana", "Armor", "MR", "DPS", "Damage", "Atk Spd", "Crit Rate", "Range"}
champ_stats = {}

for k in range(len(champ_reflist)):

    champ_name = champ_reflist[k]
    new_url = url+champ_name
    req = requests.get(new_url)
    stats = {}
    soup = BeautifulSoup(req.text, "html.parser")

    for link in soup.find_all('li'):
        for i in stat_list:
            if i in link.text:
              

                stats[i] = link.text[len(i)+2:]

    champ_stats[champ_name[:-1]] = stats
    
print(champ_stats["alistar"])



{'Cost': '1', 'Health': '650 / 1170 / 2106', 'Mana': '30 / 100', 'Armor': '40', 'MR': '40', 'DPS': '28 / 50 / 89', 'Damage': '50 / 90 / 162', 'Atk Spd': '0.55', 'Crit Rate': '25%', 'Range': '1'}


In [73]:

def totalStat(champ_stats, stat_name):

        
    for i in champ_stats:
        stat = champ_stats[i][stat_name]
        if isinstance(stat, int):
            continue
        if stat_name == "Crit Rate":
            
            champ_stats[i]["Crit Rate"] = int(stat[:-1])
            continue
        stat = stat.split(" / ")
        sum_stat = 0
        for n in stat:
            sum_stat += int(n)
    
        champ_stats[i]["Sum " + stat_name] = sum_stat

   
print(type((champ_stats.values())))
totalStat(champ_stats, "Health")
totalStat(champ_stats, "Mana")
totalStat(champ_stats, "DPS")
totalStat(champ_stats, "Damage")
totalStat(champ_stats, "Crit Rate")
print(champ_stats["alistar"])

<class 'dict_values'>
{'Cost': '1', 'Health': '650 / 1170 / 2106', 'Mana': '30 / 100', 'Armor': '40', 'MR': '40', 'DPS': '28 / 50 / 89', 'Damage': '50 / 90 / 162', 'Atk Spd': '0.55', 'Crit Rate': 25, 'Range': '1', 'Sum Health': 3926, 'Sum Mana': 130, 'Sum DPS': 167, 'Sum Damage': 302}


In [84]:
import csv
with open('tftData.csv', 'w', newline='') as csvfile:
    spamwriter = csv.writer(csvfile)
    spamwriter.writerow(['Champ Name', 'Cost', 'Health', 'Mana', 'Armour', 'Damage', 'Attack Speed', 'Crit Rate %', 'Range', 'Sum Health',
                        'Sum Mana', 'Sum DPS', 'Sum Damage'])
    for i in champ_stats:
        #print(champ_stats[i]["Cost"])
        spamwriter.writerow([i, champ_stats[i]["Cost"], champ_stats[i]["Health"], champ_stats[i]["Mana"], champ_stats[i]["Armor"], 
                             champ_stats[i]["Damage"], champ_stats[i]["Atk Spd"], champ_stats[i]["Crit Rate"], champ_stats[i]["Range"],
                            champ_stats[i]["Sum Health"], champ_stats[i]["Sum Mana"], champ_stats[i]["Sum DPS"], champ_stats[i]["Sum Damage"]]) 
    
   

In [75]:
import matplotlib.pyplot as plt
import pandas as pd
df = pd.read_csv("tftData.csv")

In [76]:
tiers = [0,0,0,1,1,1,0,1,1,0,1,0,0,0,1,1,0,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,1,1,0,1,0,0,0,1,0,0,0,0,1,1,0,1,1,0,1,0,1,1,0,0,1,1]

df["Tier"] = tiers

In [77]:
df.head()

,Champ Name,Cost,Health,Mana,Armour,Damage,Attack Speed,Crit Rate %,Range,Sum Health,Sum Mana,Sum DPS,Sum Damage,Tier
0,alistar,1,650 / 1170 / 2106,30 / 100,40,50 / 90 / 162,0.55,25,1,3926,130,167,302,0
1,annie,4,850 / 1530 / 2754,40,30,30 / 54 / 97,0.75,25,4,5134,40,137,181,0
2,aphelios,4,800 / 1440 / 2592,60,30,63 / 113 / 204,0.75,25,4,4832,60,285,380,0
3,aurora,5,800 / 1440 / 2592,20 / 80,40,50 / 90 / 162,0.80,25,4,4832,100,242,302,1
4,brand,4,800 / 1440 / 2592,25 / 75,30,35 / 63 / 113,0.75,25,4,4832,100,158,211,1


In [121]:
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
df_features = df[["Cost", "Sum Health","Sum Mana","Armour","Sum DPS", "Sum Damage", "Attack Speed", "Crit Rate %", "Range"]]
df_label = df["Tier"]
print(df_features.head())
x_train, x_test, y_train, y_test = train_test_split(
    df_features, df_label, test_size=0.3)





   Cost  Sum Health  Sum Mana  Armour  Sum DPS  Sum Damage  Attack Speed  \
0     1        3926       130      40      167         302          0.55   
1     4        5134        40      30      137         181          0.75   
2     4        4832        60      30      285         380          0.75   
3     5        4832       100      40      242         302          0.80   
4     4        4832       100      30      158         211          0.75   

   Crit Rate %  Range  
0           25      1  
1           25      4  
2           25      4  
3           25      4  
4           25      4  


In [122]:
print(y_train.value_counts())

Tier
0    22
1    20
Name: count, dtype: int64


# Built Models
 Although the accuracy for these models can be higher than 50% (even up to 80%), it can also go as low as ~30%-40%. I believe this is due to the model being "lucky" or "unlucky" with the data in the test set as changing the test_size in the train_test_split function (previous code block) can either increase or decrease the variance of the test data; leading to the model correctly or incorrectly labelling the data with more frequency, not because it has identified a pattern.  
 
## Logistic Regression

In [123]:
model = LogisticRegression(max_iter = 500).fit(x_train, y_train)
predictions = model.predict(x_test)
sklearn.metrics.accuracy_score(predictions, y_test)


0.5

## Random Forest 

In [124]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=4)
rf.fit(x_train, y_train)
rf_preds = rf.predict(x_test)

print("RF Accuracy:", sklearn.metrics.accuracy_score(y_test, rf_preds))

RF Accuracy: 0.6666666666666666
